# Bài 4 – Trang 26: Phân lớp Red Wine Quality

**6451071011 - Lê Nguyễn Anh Dự**

Chạy các ô theo thứ tự trong Google Colab. Notebook tự tải CSV từ liên kết Drive nếu chưa có tệp trong thư mục làm việc.


In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

STUDENT = '6451071011 - Lê Nguyễn Anh Dự'
FILE_ID = '1rzLiUi64RLmMV8ApudCt6aJyv_OkhADH'

def heading(number, title):
    print(STUDENT)
    print(f'Câu {number}. {title}')

def ensure_data():
    filename = 'winequality-red.csv'
    if not os.path.exists(filename):
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
            import gdown
        gdown.download(id=FILE_ID, output=filename, quiet=False)
    if not os.path.exists(filename):
        raise FileNotFoundError('Không tải được winequality-red.csv; hãy tải tệp lên Colab.')
    return filename

data = pd.read_csv(ensure_data(), sep=None, engine='python')
data.columns = data.columns.str.strip()
assert 'quality' in data.columns, 'Tệp phải có cột quality.'
print('Dữ liệu:', data.shape)
display(data.head())


## Câu 1. Tương quan Pearson

Tính Pearson giữa các thuộc tính đầu vào. Nếu hai thuộc tính có |r| ≥ 0,90, loại một thuộc tính trong cặp. Cột `quality` không tham gia lựa chọn cột cần bỏ.


In [ ]:
heading(1, 'Khảo sát tương quan Pearson')
input_data = data.drop(columns='quality')
numeric_data = input_data.select_dtypes(include=np.number)
corr = numeric_data.corr(method='pearson')
display(corr.round(3))
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Tương quan Pearson giữa các thuộc tính đầu vào')
plt.tight_layout()
plt.show()
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if (upper[col] >= 0.90).any()]
print('Cột có tương quan cao cần bỏ:', drop_corr if drop_corr else 'Không có')
input_data = input_data.drop(columns=drop_corr)


## Câu 2. Tách thuộc tính và nhãn quyết định

In [ ]:
heading(2, 'Tách X và y')
X = input_data.copy()
y = data['quality'].copy()
print('X:', X.shape, '| y:', y.shape)
print('Phân bố quality:')
display(y.value_counts().sort_index())


## Câu 3. Mã hóa one-hot các thuộc tính không phải số

In [ ]:
heading(3, 'One-hot encoding')
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
print('Cột không phải số:', categorical_columns if categorical_columns else 'Không có')
X = pd.get_dummies(X, columns=categorical_columns, dtype=np.uint8)
print('Kích thước X sau mã hóa:', X.shape)
display(X.head())


## Câu 4. Chia dữ liệu huấn luyện 70% và kiểm thử 30%

In [ ]:
heading(4, 'Chia train/test')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Huấn luyện: {len(X_train)} mẫu ({len(X_train)/len(X):.1%})')
print(f'Kiểm thử: {len(X_test)} mẫu ({len(X_test)/len(X):.1%})')


## Hàm đánh giá dùng chung

`criterion="entropy"` trong scikit-learn dùng information gain như hướng dẫn, nhưng cây này vẫn hỗ trợ chia thuộc tính số theo ngưỡng; vì vậy đây là cách mô phỏng ID3, không phải ID3 thuần túy.


In [ ]:
class_names = sorted(y.unique())
results = []

def evaluate(name, model):
    pred = model.predict(X_test)
    print(classification_report(y_test, pred, labels=class_names, zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=class_names)
    print('Ma trận nhầm lẫn (hàng: thực tế; cột: dự đoán):')
    print(cm)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.title(name)
    plt.tight_layout(); plt.show()
    results.append({'Mô hình': name,
                    'Accuracy': accuracy_score(y_test, pred),
                    'Precision macro': precision_score(y_test, pred, average='macro', zero_division=0),
                    'Recall macro': recall_score(y_test, pred, average='macro', zero_division=0),
                    'F1 macro': f1_score(y_test, pred, average='macro', zero_division=0),
                    'F1 weighted': f1_score(y_test, pred, average='weighted', zero_division=0)})

def show_tree(model, title, filename):
    print('Bốn tầng đầu của cây (mô hình được huấn luyện trên cây đầy đủ):')
    print(export_text(model, feature_names=list(X_train.columns), max_depth=3))
    plt.figure(figsize=(24, 11))
    plot_tree(model, feature_names=list(X_train.columns),
              class_names=[str(c) for c in model.classes_],
              filled=True, rounded=True, max_depth=3, fontsize=8)
    plt.title(title + ' – 4 tầng đầu')
    plt.tight_layout()
    plt.savefig(filename, dpi=100, bbox_inches='tight')
    plt.show()
    print('Đã lưu hình:', filename)


## Câu 5. Cây ID3 theo entropy

In [ ]:
heading(5, 'Cây ID3/entropy')
id3 = DecisionTreeClassifier(criterion='entropy', random_state=42)
id3.fit(X_train, y_train)
evaluate('ID3 (entropy)', id3)
show_tree(id3, 'ID3 (entropy)', 'wine_tree_id3.png')


## Câu 6. Cây CART theo Gini

In [ ]:
heading(6, 'Cây CART/Gini')
cart = DecisionTreeClassifier(criterion='gini', random_state=42)
cart.fit(X_train, y_train)
evaluate('CART (gini)', cart)
show_tree(cart, 'CART (gini)', 'wine_tree_cart.png')


## Câu 7. Mô hình Naïve Bayes

In [ ]:
heading(7, 'Gaussian Naïve Bayes')
nb = GaussianNB()
nb.fit(X_train, y_train)
evaluate('Gaussian Naïve Bayes', nb)


## Câu 8. So sánh kết quả

In [ ]:
heading(8, 'So sánh ba mô hình')
comparison = pd.DataFrame(results).set_index('Mô hình')
display(comparison.style.format('{:.4f}').highlight_max(axis=0, color='#c8e6c9'))
for metric in ['Accuracy', 'F1 macro', 'F1 weighted']:
    best = comparison[metric].idxmax()
    print(f'{metric} cao nhất: {best} ({comparison.loc[best, metric]:.4f})')
print('Nhận xét: Chất lượng rượu có nhiều mức và số mẫu giữa các mức không cân bằng; F1 macro cho biết khả năng phân loại đều các mức, còn accuracy và F1 weighted phản ánh hiệu quả tổng thể. Cây không giới hạn độ sâu có nguy cơ quá khớp. GaussianNB giả định các thuộc tính độc lập có điều kiện và phân phối Gaussian.')
